# GeoSeg × Vaihingen — Notebook de arranque rápido
Este cuaderno corre **GeoSeg** con el dataset **Vaihingen** desde el **root del repositorio**.

👉 *Solo edita la variable `DATA_ROOT` en la **Celda 2*** para apuntar a tus carpetas reales.
El flujo es:
1) Verificar entorno/GPU
2) Definir rutas (DATA_ROOT, IMG_DIR, MASK_DIR, SPLIT_DIR)
3) Sanity checks del dataset y creación de splits si faltan
4) Copiar un config base de Vaihingen y **ajustar rutas** automáticamente
5) Entrenamiento rápido (smoke test)
6) Entrenamiento completo
7) Evaluación (val/test)
8) Inferencia sobre ortofotos grandes

**Nota:** Ejecuta este cuaderno estando en la carpeta del repo (donde está `train_supervision.py`).

## 0) Comprobaciones básicas (estás en el root del repo)


In [1]:
import os, sys, glob, subprocess, json
from pathlib import Path
print('PWD:', os.getcwd())
print('Contenido:', os.listdir('.'))
assert Path('train_supervision.py').exists(), 'Ejecuta este cuaderno en el ROOT del repo GeoSeg.'

PWD: /workspace/GeoSeg
Contenido: ['.git', '.ipynb_checkpoints', 'airs', 'config', 'data', 'geoseg', 'GeoSeg_Vaihingen_Notebook.ipynb', 'inference_huge_image.py', 'inference_uavid.py', 'LICENSE', 'loveda_test.py', 'pot.png', 'potsdam_test.py', 'pretrain_weights', 'README.md', 'requirements.txt', 'tools', 'train_supervision.py', 'Untitled.ipynb', 'vai.png', 'vaihingen_test.py']


## 1) Verificar PyTorch/CUDA

In [2]:
import torch
print('CUDA disponible:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('⚠️ No se detecta GPU; el entrenamiento será muy lento.')

CUDA disponible: True
GPU: NVIDIA T1000 8GB


## 2) Define tus rutas del dataset Vaihingen
Edita **solo esta celda**. Debes tener carpetas `images/`, `masks/` y una carpeta `splits/` con `train.txt` y `val.txt`.
Si `splits/` no existe, el cuaderno puede generarlos automáticamente más abajo.

In [6]:
# ←←← EDITA AQUÍ SEGÚN TU ESTRUCTURA
from pathlib import Path

# Carpeta padre del dataset
DATA_ROOT = Path('data/vaihingen/train')   # <-- esta es la que se ve en tu panel izquierdo

# Para ENTRENAR y VALIDAR usaremos el conjunto de TRAIN (y el notebook creará split 80/20)
IMG_DIR   = DATA_ROOT / 'images_1024'   # imágenes para train/val
MASK_DIR  = DATA_ROOT / 'masks_1024'    # máscaras indexadas 0..N-1 para train/val

# Carpeta donde el cuaderno guardará/leerá las listas train.txt y val.txt
SPLIT_DIR = DATA_ROOT / 'splits'

for p in [DATA_ROOT, IMG_DIR, MASK_DIR]:
    print(p, 'existe?', p.exists())
print('SPLIT_DIR:', SPLIT_DIR)

data/vaihingen/train existe? True
data/vaihingen/train/images_1024 existe? True
data/vaihingen/train/masks_1024 existe? True
SPLIT_DIR: data/vaihingen/train/splits


## 3) Sanity checks del dataset y (opcional) generación de splits
- Verifica que los nombres base (sin extensión) coincidan entre `images/` y `masks/`.
- Si faltan `train.txt` y `val.txt`, genera un split simple 80/20.

In [7]:
from pathlib import Path
import re

img_exts = {'.tif', '.tiff', '.png', '.jpg', '.jpeg'}
mask_exts = {'.png', '.tif', '.tiff'}

imgs = sorted([p for p in IMG_DIR.iterdir() if p.suffix.lower() in img_exts])
masks = sorted([p for p in MASK_DIR.iterdir() if p.suffix.lower() in mask_exts])

print(f'#imgs: {len(imgs)}  #masks: {len(masks)}')

stem_imgs  = {p.stem for p in imgs}
stem_masks = {p.stem for p in masks}

missing_masks = sorted(stem_imgs - stem_masks)
missing_imgs  = sorted(stem_masks - stem_imgs)
print('Faltan máscaras para:', missing_masks[:10])
print('Faltan imágenes para:', missing_imgs[:10])

# Generar splits si no existen
SPLIT_DIR.mkdir(parents=True, exist_ok=True)
train_txt = SPLIT_DIR / 'train.txt'
val_txt   = SPLIT_DIR / 'val.txt'

if not train_txt.exists() or not val_txt.exists():
    print('⚠️ No se encontraron train.txt / val.txt. Creando split 80/20...')
    stems = sorted(stem_imgs & stem_masks)
    n = len(stems)
    n_train = int(0.8 * n)
    train_stems = stems[:n_train]
    val_stems   = stems[n_train:]
    train_txt.write_text('\n'.join(train_stems))
    val_txt.write_text('\n'.join(val_stems))
else:
    print('✓ Encontrados splits existentes:')
    print(' -', train_txt)
    print(' -', val_txt)

print('train.txt muestras:', sum(1 for _ in open(train_txt)))
print('val.txt muestras  :', sum(1 for _ in open(val_txt)))

#imgs: 615  #masks: 615
Faltan máscaras para: []
Faltan imágenes para: []
⚠️ No se encontraron train.txt / val.txt. Creando split 80/20...
train.txt muestras: 492
val.txt muestras  : 123


## 4) Crear un **config YAML** nuevo para Vaihingen con tus rutas
Se copiará un config base de `config/vaihingen/` y se editarán las claves de dataset más comunes.

In [11]:
# Elige el config que usarás (por ejemplo UnetFormer)
CFG_PATH = 'config/vaihingen/unetformer.py'  # o dcswin.py / ftunetformer.py

# Muestra las líneas donde suelen estar las rutas
import re, pathlib
txt = pathlib.Path(CFG_PATH).read_text(encoding='utf-8', errors='ignore')
for i, line in enumerate(txt.splitlines(), 1):
    if re.search(r'data_root|dataset_root|root|img_dir|image_dir|mask_dir|label_dir|split|train_list|val_list', line):
        print(f"{i:4d}: {line}")



  42: train_dataset = VaihingenDataset(data_root='data/vaihingen/train', mode='train',
  46: test_dataset = VaihingenDataset(data_root='data/vaihingen/test',


## 5) Entrenamiento corto (smoke test)
Ejecuta unas pocas iteraciones para comprobar que **pipeline y rutas** están OK.
Ajusta `--work_dir` si quieres separar corridas.

In [19]:
import re
from pathlib import Path

# raíz del repo dentro del contenedor
REPO = Path('/workspace/GeoSeg').resolve()

SRC = REPO / 'config' / 'vaihingen' / 'unetformer.py'
DST_DIR = REPO / 'config' / 'custom'
DST_DIR.mkdir(parents=True, exist_ok=True)
DST = DST_DIR / 'unetformer_local.py'

text = SRC.read_text(encoding='utf-8')

# 1) épocas (ajusta el 50 si quieres)
text = re.sub(r'max_epoch\s*=\s*\d+', 'max_epoch = 50', text)

# 2) evita el crash del DataLoader
text = re.sub(r'num_workers\s*=\s*\d+', 'num_workers=0', text)

# 3) baja batch si tu GPU es 8GB
text = re.sub(r'train_batch_size\s*=\s*\d+', 'train_batch_size = 2', text)
text = re.sub(r'val_batch_size\s*=\s*\d+',   'val_batch_size = 2', text)

DST.write_text(text, encoding='utf-8')

print("Escribí:", DST)
print("¿Existe?", DST.exists())
print("PWD notebook:", Path('.').resolve())



Escribí: /workspace/GeoSeg/config/custom/unetformer_local.py
¿Existe? True
PWD notebook: /workspace/GeoSeg


In [20]:
!python train_supervision.py -c /workspace/GeoSeg/config/custom/unetformer_local.py



/opt/conda/envs/airs/lib/python3.8/site-packages/albumentations/__init__.py:13: UserWarning: A new version of Albumentations is available: 2.0.8 (you have 1.4.18). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()
/opt/conda/envs/airs/lib/python3.8/site-packages/timm/models/_factory.py:117: UserWarning: Mapping deprecated model name swsl_resnet18 to current resnet18.fb_swsl_ig1b_ft_in1k.
  model = create_fn(
/opt/conda/envs/airs/lib/python3.8/site-packages/torch/functional.py:513: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at ../aten/src/ATen/native/TensorShape.cpp:3609.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/opt/conda/envs/airs/lib/python3.8/si

1) Vigilar la corrida (logs, checkpoints)

Durante el entrenamiento, el repo suele crear una carpeta de salida (p. ej. logs/, runs/ o similar). Para confirmarla y ver cómo va:

1) Verificación rápida del checkpoint

In [28]:
import torch
ckpt_path = "model_weights/vaihingen/unetformer-r18-512-crop-ms-e105/last-v1.ckpt"
ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
print("epoch:", ckpt.get("epoch"), "global_step:", ckpt.get("global_step"))
print("keys:", list(ckpt.keys())[:8])


epoch: 49 global_step: 15350
keys: ['epoch', 'global_step', 'pytorch-lightning_version', 'state_dict', 'loops', 'callbacks', 'optimizer_states', 'lr_schedulers']


2) Evaluar con tu config y guardar predicciones

Según la rama, suele existir uno de estos scripts. Corre el que tengas:

In [30]:
!python vaihingen_test.py -h



usage: vaihingen_test.py [-h] -c CONFIG_PATH -o OUTPUT_PATH [-t {None,d4,lr}]
                         [--rgb]

optional arguments:
  -h, --help            show this help message and exit
  -c CONFIG_PATH, --config_path CONFIG_PATH
                        Path to config
  -o OUTPUT_PATH, --output_path OUTPUT_PATH
                        Path where to save resulting masks.
  -t {None,d4,lr}, --tta {None,d4,lr}
                        Test time augmentation.
  --rgb                 whether output rgb images


In [47]:
!python vaihingen_test.py -c config/vaihingen/unetformer.py -o airs/fig_results/vaihingen/unetformer --rgb -t 'd4'

/opt/conda/envs/airs/lib/python3.8/site-packages/albumentations/__init__.py:13: UserWarning: A new version of Albumentations is available: 2.0.8 (you have 1.4.18). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()
/opt/conda/envs/airs/lib/python3.8/site-packages/timm/models/_factory.py:117: UserWarning: Mapping deprecated model name swsl_resnet18 to current resnet18.fb_swsl_ig1b_ft_in1k.
  model = create_fn(
/opt/conda/envs/airs/lib/python3.8/site-packages/torch/functional.py:513: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at ../aten/src/ATen/native/TensorShape.cpp:3609.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
/opt/conda/envs/airs/lib/python3.8/site-packages/lightning_fabric/utilities/cloud_io.py:57: You are using `torch.load` with `weights_only=False` (the cur

In [48]:
!python inference_huge_image.py \
-i data/vaihingen/test_images \
-c GeoSeg/config/vaihingen/dcswin.py \
-o fig_results/vaihingen/dcswin_huge \
-t 'lr' -ph 512 -pw 512 -b 2 -d "pv"

/opt/conda/envs/airs/lib/python3.8/site-packages/albumentations/__init__.py:13: UserWarning: A new version of Albumentations is available: 2.0.8 (you have 1.4.18). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()
Traceback (most recent call last):
  File "inference_huge_image.py", line 10, in <module>
    from catalyst.dl import SupervisedRunner
ModuleNotFoundError: No module named 'catalyst'


In [50]:
!python vaihingen_test.py \
  -c /workspace/GeoSeg/config/vaihingen/unetformer.py \
  -o airs/fig_results/vaihingen/unetformer \
  --rgb -t d4

/opt/conda/envs/airs/lib/python3.8/site-packages/albumentations/__init__.py:13: UserWarning: A new version of Albumentations is available: 2.0.8 (you have 1.4.18). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()
/opt/conda/envs/airs/lib/python3.8/site-packages/timm/models/_factory.py:117: UserWarning: Mapping deprecated model name swsl_resnet18 to current resnet18.fb_swsl_ig1b_ft_in1k.
  model = create_fn(
/opt/conda/envs/airs/lib/python3.8/site-packages/torch/functional.py:513: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at ../aten/src/ATen/native/TensorShape.cpp:3609.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
/opt/conda/envs/airs/lib/python3.8/site-packages/lightning_fabric/utilities/cloud_io.py:57: You are using `torch.load` with `weights_only=False` (the cur

In [66]:
!find airs/fig_results/vaihingen/unetformer -maxdepth 2 -type f | head -n 20


In [55]:
from tools.cfg import py2cfg
import os, re, pprint

CFG_PATH = "/workspace/GeoSeg/config/vaihingen/unetformer.py"   # <-- la que usaste
cfg = py2cfg(CFG_PATH)

print(">> Claves relevantes en config:")
for k in sorted(cfg.keys()):
    if re.search(r"(weight|ckpt|path|root|dir|data|vai)", k, re.I):
        print(f"  {k}: {cfg[k]}")

# ¿Qué .ckpt intentará cargar el test?
ckpt_path = os.path.join(cfg.get("weights_path",""), cfg.get("test_weights_name","") + ".ckpt")
print("\n>> Checkpoint que intentará cargar:", ckpt_path)
print("   Existe?:", os.path.exists(ckpt_path))



>> Claves relevantes en config:
  DataLoader: <class 'torch.utils.data.dataloader.DataLoader'>
  Dataset: <class 'torch.utils.data.dataset.Dataset'>
  VaihingenDataset: <class 'geoseg.datasets.vaihingen_dataset.VaihingenDataset'>
  WeightedLoss: <class 'geoseg.losses.joint_loss.WeightedLoss'>
  backbone_weight_decay: 0.01
  pretrained_ckpt_path: None
  resume_ckpt_path: None
  test_dataset: <geoseg.datasets.vaihingen_dataset.VaihingenDataset object at 0x7f24bb26ea90>
  test_weights_name: unetformer-r18-512-crop-ms-e105
  train_dataset: <geoseg.datasets.vaihingen_dataset.VaihingenDataset object at 0x7f24bb26ebb0>
  val_dataset: <geoseg.datasets.vaihingen_dataset.VaihingenDataset object at 0x7f24bb26eaf0>
  weight_decay: 0.01
  weights_name: unetformer-r18-512-crop-ms-e105
  weights_path: model_weights/vaihingen/unetformer-r18-512-crop-ms-e105

>> Checkpoint que intentará cargar: model_weights/vaihingen/unetformer-r18-512-crop-ms-e105/unetformer-r18-512-crop-ms-e105.ckpt
   Existe?: True

In [56]:
!mkdir -p airs/fig_results/vaihingen/unetformer

!python vaihingen_test.py \
  -c /workspace/GeoSeg/config/vaihingen/unetformer.py \
  -o airs/fig_results/vaihingen/unetformer \
  -t d4 --rgb 2>&1 | tee /tmp/vaih_test.log


/opt/conda/envs/airs/lib/python3.8/site-packages/albumentations/__init__.py:13: UserWarning: A new version of Albumentations is available: 2.0.8 (you have 1.4.18). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()
/opt/conda/envs/airs/lib/python3.8/site-packages/timm/models/_factory.py:117: UserWarning: Mapping deprecated model name swsl_resnet18 to current resnet18.fb_swsl_ig1b_ft_in1k.
  model = create_fn(
/opt/conda/envs/airs/lib/python3.8/site-packages/torch/functional.py:513: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at ../aten/src/ATen/native/TensorShape.cpp:3609.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
/opt/conda/envs/airs/lib/python3.8/site-packages/lightning_fabric/utilities/cloud_io.py:57: You are using `torch.load` with `weights_only=False` (the cur

In [57]:
!echo "---- ERRORES ----"; grep -nE "Traceback|Error|No such file|size mismatch|RuntimeError" /tmp/vaih_test.log || echo "(sin errores)"
!echo "---- GUARDADOS ----"; grep -nEi "save|writing|pred|output|mask" /tmp/vaih_test.log || echo "(sin mensajes de guardado)"


---- ERRORES ----
(sin errores)
---- GUARDADOS ----
24:images writing spends: 0.16385531425476074 s


In [60]:
# listar archivos guardados (preds/rgb suelen ser subcarpetas)
!find airs/fig_results/vaihingen/unetformer -maxdepth 3 -type f \( -iname "*.png" -o -iname "*.tif" -o -iname "*.jpg" \) | sort | head -n 50


In [62]:
!pwd
!ls -ld airs/fig_results/vaihingen/unetformer || echo "NO EXISTE"


/workspace/GeoSeg
drwxr-xr-x 1 root root 512 Oct 29 04:43 airs/fig_results/vaihingen/unetformer


In [63]:
OUT="/workspace/GeoSeg/airs/fig_results/vaihingen/unetformer"
!mkdir -p "$OUT"

!python vaihingen_test.py \
  -c /workspace/GeoSeg/config/vaihingen/unetformer.py \
  -o "$OUT" \
  -t d4 --rgb 2>&1 | tee /tmp/vaih_test2.log


/opt/conda/envs/airs/lib/python3.8/site-packages/albumentations/__init__.py:13: UserWarning: A new version of Albumentations is available: 2.0.8 (you have 1.4.18). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()
/opt/conda/envs/airs/lib/python3.8/site-packages/timm/models/_factory.py:117: UserWarning: Mapping deprecated model name swsl_resnet18 to current resnet18.fb_swsl_ig1b_ft_in1k.
  model = create_fn(
/opt/conda/envs/airs/lib/python3.8/site-packages/torch/functional.py:513: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at ../aten/src/ATen/native/TensorShape.cpp:3609.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
/opt/conda/envs/airs/lib/python3.8/site-packages/lightning_fabric/utilities/cloud_io.py:57: You are using `torch.load` with `weights_only=False` (the cur

In [64]:
OUT="/workspace/GeoSeg/airs/fig_results/vaihingen/unetformer"
!echo "Mostrando archivos en $OUT"
!find "$OUT" -maxdepth 3 -type f \( -iname "*.png" -o -iname "*.tif" -o -iname "*.jpg" -o -iname "*.bmp" -o -iname "*.npy" \) | sort | head -n 50


Mostrando archivos en /workspace/GeoSeg/airs/fig_results/vaihingen/unetformer


In [65]:
!find /workspace/GeoSeg -type f -mmin -10 \( -iname "*.png" -o -iname "*.tif" -o -iname "*.jpg" -o -iname "*.bmp" -o -iname "*.npy" \) | head -n 80


In [34]:
from pathlib import Path, re

CFG  = Path("/workspace/GeoSeg/config/custom/unetformer_local.py")
CKPT = "model_weights/vaihingen/unetformer-r18-512-crop-ms-e105"

txt = CFG.read_text(encoding="utf-8", errors="ignore")
# usa la variable ckpt_path en la config (créala o actualízala)
if not re.search(r"^\s*ckpt_path\s*=", txt, re.M):
    txt += f'\n# añadido para test\nckpt_path = r"{CKPT}"\n'
else:
    txt = re.sub(r'^\s*ckpt_path\s*=.*$', f'ckpt_path = r"{CKPT}"', txt, flags=re.M)
CFG.write_text(txt, encoding="utf-8")
print("ckpt_path ->", CKPT)


ckpt_path -> model_weights/vaihingen/unetformer-r18-512-crop-ms-e105


!find work_dirs/vaihingen/unetformer_r18_512_crop_ms_e105/preds_lastv1_d4 -maxdepth 2 -type f | head -n 20

## 7) Evaluación / Test
Apunta al `best.pth` o `last.pth` que quede en `runs/.../checkpoints/`. Ajusta `CKPT` y `SAVE_DIR`.

In [ ]:
CKPT = 'runs/vaih_debug/checkpoints/best.pth'  # <-- ajusta si usas otro work_dir
SAVE_DIR = 'outputs/vaih_eval'
cmd = f"python vaihingen_test.py --config {CUSTOM_YAML} --ckpt {CKPT} --save_dir {SAVE_DIR}"
print('CMD:', cmd)

*(Ejecuta la celda de arriba para ver el comando; luego quítale el comentario aquí abajo para correrlo)*

In [ ]:
# import subprocess, shlex
# subprocess.run(shlex.split(cmd), check=False)